
# Preprocesamiento de datos
### Enfoque para alumnos de Ingeniería Mecatrónica

Este notebook introduce los conceptos fundamentales del **preprocesamiento de datos** usando Python, con ejemplos relacionados con **sensores, variables de proceso, actuadores y sistemas mecatrónicos**.

## Objetivos de aprendizaje
Al finalizar este material, el alumno será capaz de:

- comprender por qué el preprocesamiento es una etapa crítica antes del modelado;
- identificar datos faltantes, duplicados y valores atípicos;
- limpiar y transformar un conjunto de datos;
- escalar variables numéricas;
- codificar variables categóricas;
- preparar datos para tareas básicas de aprendizaje automático.

## Herramientas utilizadas
En este notebook trabajaremos con:

- `pandas` para manipulación tabular de datos,
- `numpy` para operaciones numéricas,
- `matplotlib` para visualización básica.



## 1. ¿Qué es el preprocesamiento de datos?

El **preprocesamiento** consiste en preparar los datos antes de entrenar un modelo de Machine Learning o antes de realizar un análisis formal.

En un contexto de Ingeniería Mecatrónica, esto puede incluir:

- registros de temperatura, corriente, vibración o posición;
- datos provenientes de PLC, microcontroladores o sistemas SCADA;
- mediciones experimentales capturadas en laboratorio;
- variables de calidad o producción en un proceso automatizado.

### Problemas comunes en los datos
En la práctica, los datos suelen tener problemas como:

- valores faltantes,
- filas duplicadas,
- errores de captura,
- diferentes escalas entre variables,
- ruido,
- valores extremos,
- etiquetas inconsistentes.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)



## 2. Crear un conjunto de datos de ejemplo

Vamos a simular un pequeño conjunto de datos de un sistema mecatrónico. Supón que estamos monitoreando un motor eléctrico con sensores de:

- temperatura,
- vibración,
- corriente,
- velocidad,
- estado del sistema.

Después introduciremos algunos errores intencionales para practicar el preprocesamiento.


In [ ]:

n = 30

df = pd.DataFrame({
    "temperatura_C": np.random.normal(45, 4, n).round(2),
    "vibracion_mm_s": np.random.normal(3.2, 0.8, n).round(2),
    "corriente_A": np.random.normal(8.5, 1.2, n).round(2),
    "velocidad_rpm": np.random.normal(1450, 60, n).round(0),
    "modo_operacion": np.random.choice(["manual", "automatico"], n),
    "estado_motor": np.random.choice(["normal", "alerta"], n, p=[0.8, 0.2])
})

df.head()



## 3. Introducir problemas comunes en los datos

Añadiremos algunos casos típicos:

- valores faltantes,
- fila duplicada,
- valor atípico,
- categorías con distinta escritura.


In [ ]:

df.loc[3, "temperatura_C"] = np.nan
df.loc[8, "corriente_A"] = np.nan
df.loc[12, "modo_operacion"] = "Manual"   # inconsistencia
df.loc[15, "temperatura_C"] = 95          # valor atípico
df = pd.concat([df, df.iloc[[5]]], ignore_index=True)  # duplicado

df.tail()



## 4. Inspección inicial de los datos

Antes de limpiar, siempre conviene revisar:

- tamaño del DataFrame,
- tipos de datos,
- resumen estadístico,
- cantidad de valores faltantes.


In [ ]:

print("Dimensiones:", df.shape)
print("\nTipos de datos:")
print(df.dtypes)


In [ ]:

df.describe(include="all")


In [ ]:

df.isnull().sum()



## 5. Tratamiento de valores faltantes

Los valores faltantes (`NaN`) pueden aparecer por:

- fallas de comunicación,
- errores de lectura del sensor,
- pérdida de paquetes,
- interrupciones del sistema.

### Estrategias comunes
Algunas opciones son:

1. eliminar filas con datos faltantes;
2. reemplazar por la media;
3. reemplazar por la mediana;
4. interpolar;
5. usar conocimiento del proceso.

Primero haremos una copia para no perder el original.


In [ ]:

df_limpio = df.copy()



### 5.1 Imputar valores numéricos con la mediana

La **mediana** suele ser útil cuando podrían existir valores atípicos.


In [ ]:

for col in ["temperatura_C", "corriente_A", "vibracion_mm_s", "velocidad_rpm"]:
    df_limpio[col] = df_limpio[col].fillna(df_limpio[col].median())

df_limpio.isnull().sum()



## 6. Corregir inconsistencias en variables categóricas

En datos reales es común encontrar problemas como:

- `Manual`, `manual`, `MANUAL`,
- `OK`, `ok`, `Ok`.

Esto provoca categorías duplicadas que en realidad representan lo mismo.


In [ ]:

df_limpio["modo_operacion"].unique()


In [ ]:

df_limpio["modo_operacion"] = df_limpio["modo_operacion"].str.lower().str.strip()
df_limpio["modo_operacion"].unique()



## 7. Eliminar duplicados

Si un registro aparece repetido, puede sesgar los resultados del análisis y del modelo.


In [ ]:

print("Filas antes de eliminar duplicados:", len(df_limpio))
df_limpio = df_limpio.drop_duplicates()
print("Filas después de eliminar duplicados:", len(df_limpio))



## 8. Detección de valores atípicos

Un valor atípico puede deberse a:

- una condición real de falla,
- un error del sensor,
- ruido,
- una captura anómala.

No siempre deben eliminarse automáticamente. Primero deben **interpretarse**.

Veamos la distribución de la temperatura.


In [ ]:

plt.figure(figsize=(8,4))
plt.hist(df_limpio["temperatura_C"], bins=10, edgecolor="black")
plt.title("Distribución de temperatura")
plt.xlabel("Temperatura (°C)")
plt.ylabel("Frecuencia")
plt.grid(alpha=0.3)
plt.show()


In [ ]:

plt.figure(figsize=(6,4))
plt.boxplot(df_limpio["temperatura_C"])
plt.title("Boxplot de temperatura")
plt.ylabel("Temperatura (°C)")
plt.grid(alpha=0.3)
plt.show()



### 8.1 Método del rango intercuartílico (IQR)

Una forma sencilla de detectar valores atípicos es usar:

- `Q1`: primer cuartil,
- `Q3`: tercer cuartil,
- `IQR = Q3 - Q1`.

Se consideran atípicos los valores menores que:

- `Q1 - 1.5*IQR`

o mayores que:

- `Q3 + 1.5*IQR`


In [ ]:

Q1 = df_limpio["temperatura_C"].quantile(0.25)
Q3 = df_limpio["temperatura_C"].quantile(0.75)
IQR = Q3 - Q1

lim_inf = Q1 - 1.5 * IQR
lim_sup = Q3 + 1.5 * IQR

print("Límite inferior:", round(lim_inf, 2))
print("Límite superior:", round(lim_sup, 2))

outliers = df_limpio[(df_limpio["temperatura_C"] < lim_inf) | (df_limpio["temperatura_C"] > lim_sup)]
outliers



### 8.2 Filtrar valores atípicos

En este ejemplo eliminaremos los atípicos de temperatura, aunque en un caso real primero se debe justificar técnicamente.


In [ ]:

df_limpio = df_limpio[(df_limpio["temperatura_C"] >= lim_inf) & (df_limpio["temperatura_C"] <= lim_sup)]
df_limpio.shape



## 9. Escalamiento de variables numéricas

Muchos algoritmos de ML funcionan mejor cuando las variables numéricas tienen escalas comparables.

Por ejemplo:

- temperatura en °C,
- velocidad en rpm,
- corriente en A.

Si una variable tiene valores muy grandes respecto a otra, podría dominar el entrenamiento.

### Dos técnicas comunes
1. **Normalización Min-Max**: lleva los valores a un rango entre 0 y 1.
2. **Estandarización**: transforma para tener media 0 y desviación estándar 1.



### 9.1 Normalización Min-Max manual


In [ ]:

columnas_num = ["temperatura_C", "vibracion_mm_s", "corriente_A", "velocidad_rpm"]

df_minmax = df_limpio.copy()

for col in columnas_num:
    minimo = df_minmax[col].min()
    maximo = df_minmax[col].max()
    df_minmax[col] = (df_minmax[col] - minimo) / (maximo - minimo)

df_minmax.head()



### 9.2 Estandarización manual


In [ ]:

df_std = df_limpio.copy()

for col in columnas_num:
    media = df_std[col].mean()
    std = df_std[col].std()
    df_std[col] = (df_std[col] - media) / std

df_std.head()



## 10. Codificación de variables categóricas

Los modelos numéricos no entienden directamente palabras como:

- `manual`,
- `automatico`,
- `normal`,
- `alerta`.

Por eso es necesario convertir las variables categóricas en números.

### Opción común: One-Hot Encoding
Se crea una columna para cada categoría.


In [ ]:

df_codificado = pd.get_dummies(df_limpio, columns=["modo_operacion", "estado_motor"], drop_first=False)
df_codificado.head()



## 11. Selección de variables de entrada y salida

Supón que queremos predecir si el motor se encuentra en estado de alerta a partir de variables sensadas.


In [ ]:

df_modelo = df_limpio.copy()

# Convertir variable objetivo a binaria
df_modelo["estado_alerta"] = df_modelo["estado_motor"].map({"normal": 0, "alerta": 1})

# Codificar modo de operación
df_modelo = pd.get_dummies(df_modelo, columns=["modo_operacion"], drop_first=True)

X = df_modelo.drop(columns=["estado_motor", "estado_alerta"])
y = df_modelo["estado_alerta"]

print("Variables de entrada:")
print(X.head())
print("\nVariable objetivo:")
print(y.head())



## 12. División básica de datos: entrenamiento y prueba

En Machine Learning es importante separar los datos para evaluar el desempeño del modelo.

Una división común es:

- 80% para entrenamiento,
- 20% para prueba.


In [ ]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)



## 13. Flujo general de preprocesamiento

Un flujo típico podría ser:

1. cargar datos;
2. inspeccionar estructura y calidad;
3. tratar valores faltantes;
4. corregir categorías;
5. eliminar duplicados;
6. detectar y tratar valores atípicos;
7. escalar variables;
8. codificar categorías;
9. separar variables de entrada y salida;
10. dividir en entrenamiento y prueba.

Este flujo no siempre es rígido, pero ayuda a mantener orden y trazabilidad.



## 14. Ejemplo aplicado: señales de un actuador lineal

Generemos otro ejemplo corto para reforzar la idea. Supón que medimos un actuador lineal y registramos:

- posición,
- corriente,
- error de control.


In [ ]:

datos_actuador = pd.DataFrame({
    "posicion_mm": [10, 12, 14, 15, 16, np.nan, 18, 19, 19, 100],
    "corriente_A": [1.2, 1.3, 1.5, 1.4, 1.6, 1.5, np.nan, 1.7, 1.7, 3.9],
    "error_mm": [0.5, 0.4, 0.3, 0.2, 0.3, 0.4, 0.2, 0.1, 0.1, 5.0]
})

datos_actuador


In [ ]:

# Imputación simple con mediana
for col in datos_actuador.columns:
    datos_actuador[col] = datos_actuador[col].fillna(datos_actuador[col].median())

datos_actuador


In [ ]:

plt.figure(figsize=(7,4))
plt.plot(datos_actuador["posicion_mm"], marker="o", label="Posición")
plt.plot(datos_actuador["corriente_A"], marker="s", label="Corriente")
plt.title("Variables del actuador")
plt.xlabel("Muestra")
plt.ylabel("Valor")
plt.legend()
plt.grid(alpha=0.3)
plt.show()



## 15. Buenas prácticas en preprocesamiento

- No eliminar datos sin justificación.
- Documentar cada transformación.
- Conservar una copia del dataset original.
- Revisar unidades y rangos físicos posibles.
- Validar con conocimiento de ingeniería, no solo con fórmulas estadísticas.
- Evitar fuga de información al escalar o imputar usando datos de prueba.



# Ejercicios propuestos

## Ejercicio 1. Identificación de problemas
Crea un DataFrame con 20 registros de un sensor de temperatura y agrega intencionalmente:

- 2 valores faltantes,
- 1 fila duplicada,
- 1 valor extremo.

Después identifica cada problema con funciones de `pandas`.

---

## Ejercicio 2. Limpieza básica
A partir de un DataFrame con columnas:

- `voltaje_V`,
- `corriente_A`,
- `potencia_W`,
- `modo_operacion`,

realiza lo siguiente:

1. corrige inconsistencias de texto en `modo_operacion`;
2. rellena valores faltantes con la mediana;
3. elimina duplicados.

---

## Ejercicio 3. Detección de outliers
Genera 30 datos de vibración de un motor y agrega 2 valores muy altos.  
Usa el método de IQR para detectar los valores atípicos y muéstralos en pantalla.

---

## Ejercicio 4. Escalamiento
Con un conjunto de datos de posición, velocidad y aceleración:

1. aplica normalización Min-Max,
2. aplica estandarización,
3. compara ambos resultados.

---

## Ejercicio 5. Codificación categórica
Crea una tabla con las columnas:

- `tipo_sensor` (`temperatura`, `proximidad`, `presion`),
- `zona` (`entrada`, `proceso`, `salida`).

Convierte estas variables categóricas en formato numérico usando `pd.get_dummies()`.

---

## Ejercicio 6. Mini proyecto de mecatrónica
Simula un conjunto de datos de un sistema de monitoreo industrial con las columnas:

- `temperatura_C`,
- `corriente_A`,
- `vibracion_mm_s`,
- `velocidad_rpm`,
- `estado`.

Realiza un flujo completo de preprocesamiento que incluya:

- inspección,
- valores faltantes,
- duplicados,
- outliers,
- codificación,
- escalamiento.

Al final, deja listo el dataset para entrenar un modelo de ML.

---

## Ejercicio 7. Reflexión técnica
Responde brevemente:

1. ¿Por qué no siempre conviene eliminar un valor atípico?
2. ¿Por qué es importante escalar variables antes de ciertos modelos?
3. ¿Qué riesgo existe si se mezclan datos de entrenamiento y prueba durante el preprocesamiento?



## Cierre

El preprocesamiento de datos es una etapa fundamental en cualquier proyecto de análisis o Machine Learning.  
En Ingeniería Mecatrónica, preparar correctamente los datos permite obtener modelos más confiables, diagnósticos más claros y mejores decisiones sobre sensores, actuadores y procesos automatizados.

La calidad del modelo casi siempre depende, en gran medida, de la calidad del preprocesamiento.
